# Chapter 4 — Routing Agent

## 학습 목표
1. 질의를 **4축** (intent / entity / topic / rewriting) 으로 augment.
2. **3-backend 병렬 검색** (vector / fulltext / Cypher) + **RRF aggregation**.
3. **인용 의무 + 거절 계약** 으로 fabrication 차단.
4. 4-provider 가 같은 질의에 대해 어떤 라우팅 결정을 내리는지 Opik 으로 비교.

In [ ]:
from pathlib import Path
import os, sys, json, asyncio
from dotenv import load_dotenv

HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False); break

from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik
from _shared.providers import compare_providers, available_providers, chat, PROVIDERS
from _shared.finder_loader import sample_per_category

trace_info = setup_opik('04', only_jsonl=not os.getenv('OPIK_API_KEY'))
print('Opik project:', trace_info['project'])
USE_NEO4J = bool(os.getenv('NEO4J_PASSWORD'))

## 4.1 4-axis Question Augmentation

각 axis를 **별 호출**로 분리 → Opik trace에서 라우팅 결정 근거를 따로 추적할 수 있다.

| Axis | 출력 | 후속 결정에 미치는 영향 |
|---|---|---|
| Intent classification | lookup / aggregation / comparison / explanation | backend 가중치 |
| Entity extraction | ontology class로 lift된 명명 개체 | Cypher 패턴 결정 |
| Topic mapping | community_id 후보 | 검색 범위 좁히기 |
| Query rewriting | sub-question 분해 | multi-hop 처리 |

In [ ]:
AUG_SCHEMAS = {
    'intent': (
        'Classify the user question into ONE of: lookup, aggregation, comparison, explanation. '
        'Output JSON {"intent": <one>, "confidence": 0..1, "reason": short str}.'
    ),
    'entity': (
        'Extract named entities from the question and tag each with a FIBO class '
        '(Company/Risk/Filing/Executive/None). '
        'Output JSON {"entities": [{"text": str, "class": str}]}.'
    ),
    'topic': (
        'Map the question to up to 3 likely community_id values (integers). '
        'You do not have to be exact. Output JSON {"communities": [int], "reason": str}.'
    ),
    'rewrite': (
        'Rewrite the question to: (a) resolve pronouns/ellipsis, (b) split multi-hop into sub-questions. '
        'Output JSON {"clean": str, "sub_questions": [str]}.'
    ),
}

def augment(question: str, provider: str = 'openai') -> dict:
    out = {}
    for axis, schema in AUG_SCHEMAS.items():
        resp = chat(provider, user=question, system=schema, temperature=0.1, max_tokens=300)
        try:
            out[axis] = json.loads(resp.text)
        except Exception:
            out[axis] = {'raw': resp.text}
    return out

Q = '애플과 마이크로소프트가 공통으로 보고한 위험 요인 중 사이버보안 관련 항목을 알려줘.'
if available_providers().get('openai'):
    aug = augment(Q)
    print(json.dumps(aug, ensure_ascii=False, indent=2))
else:
    print('OPENAI_API_KEY 필요')

## 4.2 Routing Table — 3-backend 가중치

| Intent | Vector | Fulltext | Cypher | 비고 |
|---|---|---|---|---|
| `lookup` (entity 식별) | low | low | **high** | Cypher 단독 충분 |
| `lookup` (entity 모호) | mid | **high** | mid | fulltext로 후보 좁힌 후 Cypher |
| `aggregation` | low | low | **high** | GDS 도구 결합 가능 |
| `comparison` | **high** | **high** | **high** | 3개 모두 + rerank |
| `explanation` | **high** | mid | low | vector + 보조 fulltext |

In [ ]:
ROUTING = {
    'lookup'      : {'vector': 0.2, 'fulltext': 0.3, 'cypher': 1.0},
    'aggregation' : {'vector': 0.0, 'fulltext': 0.0, 'cypher': 1.0},
    'comparison'  : {'vector': 0.8, 'fulltext': 0.8, 'cypher': 0.8},
    'explanation' : {'vector': 1.0, 'fulltext': 0.4, 'cypher': 0.0},
}

def decide_backends(aug: dict) -> dict:
    intent = (aug.get('intent') or {}).get('intent', 'explanation')
    return ROUTING.get(intent, ROUTING['explanation'])

### 📎 Routing Decision Design — `chapter-04-routing-decision-design.md`

본편의 routing table 은 *what*만 보여준다. 부속 문서는 *how*와 *why* 를 정리한다:
- 전체 의사결정 트리 (intent → topic → confidence gate → weights)
- 측정 가능한 confidence threshold 표 + fail-safe 방향
- 모델별 context window budget + adaptive top-N
- **Temporal staleness penalty** (Ch 1 §10 연계) — exponential decay 로 confidence 곱셈
- 거절(refusal) 결정 트리 + Opik에 거절 사유 metadata

부속 문서: [`chapter-04-routing-decision-design.md`](./chapter-04-routing-decision-design.md)

**원칙**: *라우팅 정확도는 모델이 아니라 측정 가능한 임계값과 명시적 결정 트리에서 온다.*

In [ ]:
# 부속 문서 §4: temporal staleness penalty — exponential decay
import math
from datetime import datetime, timedelta

def staleness_penalty(published_at: datetime | None, *, half_life_days: float = 180) -> float:
    """published_at 이 신선할수록 1.0 에 가까움. half_life_days 마다 절반."""
    if published_at is None:
        return 1.0
    days = (datetime.utcnow() - published_at).days
    if days <= 0:
        return 1.0
    return math.exp(-math.log(2) * days / half_life_days)

# 데모: 같은 confidence 0.9 가 신선도에 따라 어떻게 깎이는지
raw = 0.90
for days_old, label in [(0, '오늘'), (30, '1개월'), (180, '6개월'), (365, '1년'), (730, '2년')]:
    pub = datetime.utcnow() - timedelta(days=days_old)
    pen = staleness_penalty(pub)
    eff = raw * pen
    print(f'  {label:6s} (age={days_old:4d}d): penalty={pen:.3f}  effective_confidence={eff:.3f}')

## 4.3 Parallel Search + RRF

각 백엔드의 score scale이 다르므로 raw score 합산은 위험. **Reciprocal Rank Fusion**은 rank만 보므로 robust.

$$\mathrm{RRF}(d) = \sum_{b \in \text{backends}} \frac{1}{k + \mathrm{rank}_b(d)}$$

보통 $k = 60$.

In [ ]:
from collections import defaultdict

def reciprocal_rank_fusion(per_backend_results: dict[str, list[dict]], *, k: int = 60, top_n: int = 10) -> list[dict]:
    """각 백엔드의 ranked list를 RRF로 합산.

    per_backend_results: {backend_name: [{'doc_id': str, ...}, ...]}
    """
    scores: dict[str, float] = defaultdict(float)
    payloads: dict[str, dict] = {}
    for backend, ranked in per_backend_results.items():
        for rank, item in enumerate(ranked, start=1):
            doc_id = item['doc_id']
            scores[doc_id] += 1.0 / (k + rank)
            payloads.setdefault(doc_id, item)
    fused = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_n]
    return [{**payloads[d], 'rrf': s} for d, s in fused]

# 데모: 3 backend가 같은 도큐먼트 풀에서 다른 순위를 매겼을 때
mock = {
    'vector'  : [{'doc_id': 'A'}, {'doc_id': 'B'}, {'doc_id': 'C'}, {'doc_id': 'D'}],
    'fulltext': [{'doc_id': 'C'}, {'doc_id': 'A'}, {'doc_id': 'E'}, {'doc_id': 'B'}],
    'cypher'  : [{'doc_id': 'A'}, {'doc_id': 'F'}, {'doc_id': 'C'}],
}
print('RRF top-5:')
for r in reciprocal_rank_fusion(mock, top_n=5):
    print(f"  {r['doc_id']}  rrf={r['rrf']:.4f}")

## 4.4 Citation Envelope + 거절 계약

**거절 계약 (refusal contract)** — 컨텍스트에 근거 없으면 fabrication 금지, 표준 거절 응답.

**Output Envelope**
```
<answer>...본문 with [src:chunk] 인용...</answer>
<citations>
  - src:abc, chunk:12 — "원문 인용"
</citations>
<confidence>high | medium | low</confidence>
```

In [ ]:
ANSWER_SYSTEM = (
    'You answer financial questions using ONLY the provided context.\n'
    'RULES:\n'
    '1. Cite every factual sentence with [src:chunk] inline.\n'
    '2. If the context does NOT contain enough evidence, refuse: '
    '"제공된 컨텍스트에 근거가 없습니다. 다음을 시도하세요: ..."\n'
    '3. Wrap output as <answer>...</answer><citations>...</citations><confidence>high|medium|low</confidence>.\n'
    '4. Do not invent citations. Cite only ids that appear in context.'
)

mock_context = '''
[src:finder-A, chunk:1] Apple Inc. reports cybersecurity as a top risk factor due to nation-state actors.
[src:finder-A, chunk:2] The company invests in supply chain audits.
[src:finder-M, chunk:5] Microsoft notes ransomware and supply-chain attacks among cyber risks.
'''

df = compare_providers(
    user=f'Context:\n{mock_context}\n\nQuestion: 애플과 MS가 공통으로 보고한 사이버 위험을 한 줄로.',
    system=ANSWER_SYSTEM, max_tokens=400, temperature=0.0,
)
for _, r in df.iterrows():
    print(f"--- {r['provider']} ---")
    print(r['response'][:400])
    print()

### 인용 검증

답변에 등장한 `[src:chunk]` 토큰이 실제로 컨텍스트에 있는지 자동 점검.

In [ ]:
import re
CITE_RE = re.compile(r'\[src:([^,\]]+),\s*chunk:(\d+)\]')

def validate_citations(answer: str, context: str) -> dict:
    found = set(CITE_RE.findall(answer))
    valid = set(CITE_RE.findall(context))
    return {
        'cited': sorted(found),
        'invalid': sorted(found - valid),
        'all_valid': found.issubset(valid),
    }

for _, r in df.iterrows():
    v = validate_citations(r['response'], mock_context)
    print(f"{r['provider']:9s} cited={v['cited']}  invalid={v['invalid']}  ok={v['all_valid']}")

**관찰**: invalid 인용이 0이어야 정상. provider별 inv 카운트 = fabrication 신호 → Ch 5의 self-reflection 또는 debate로 보정.

## 4.5 End-to-end Routing Demo (mock)

augment → decide_backends → search(mock) → RRF → answer

In [ ]:
if available_providers().get('openai'):
    aug = augment(Q)
    weights = decide_backends(aug)
    print('augmentation:'); print(json.dumps(aug, ensure_ascii=False, indent=2)[:600])
    print('\nrouting weights:', weights)
    fused = reciprocal_rank_fusion(mock, top_n=3)
    print('fused top-3:', fused)
    # 실제 환경에서는 fused 결과의 문서 본문을 컨텍스트로 주입.
    print('\n(end-to-end: 실제 검색 백엔드 연결 시 fused → answer cell 로 이어진다)')

## 4.6 GraphAgenticLoop — route → execute → evaluate → improve

지금까지 본 RoutingPolicy + RRF + citation envelope 를 하나의 *피드백 루프*로 묶으면 multi-agent 가 완성된다. `seocho.agents.GraphAgenticLoop` 는:

1. **augment** — intent / entities 추출 (heuristic)
2. **route** — `RoutingPolicy.decide()` → backend weights
3. **execute** — `client.ask()` 로 실제 검색·합성
4. **evaluate** — empty / citation / confidence 평가
5. **improve** — 부족하면 `reasoning_mode` 켜기, repair budget 늘리기, citation 강제
6. **stop** — high confidence 또는 변화 없음 또는 max_iterations

각 iteration 의 결정 근거가 `LoopIteration` 객체에 그대로 보존되어 Opik trace 와 짝 맞춰 디버깅 가능.

In [ ]:
# === GraphAgenticLoop 데모 ===
# 전제: Ch 1 에서 인덱싱한 그래프가 있어야 의미 있는 결과.
from seocho.agents import GraphAgenticLoop
from seocho.routing import RoutingPolicy

loop = GraphAgenticLoop(
    client,                       # ch01 에서 만든 Seocho 인스턴스
    policy=RoutingPolicy.default(),
    max_iterations=3,
)

result = loop.run('Which companies report cybersecurity as a risk factor?')
print(result.summary())
print()
print('=== final answer ===')
print(result.final_answer[:600])


In [ ]:
# 각 iteration 의 결정 trace — Opik metadata 와 짝 맞춰 본다
import json as _json
for it in result.iterations:
    print(_json.dumps(it.to_dict(), indent=2, ensure_ascii=False)[:800])
    print('---')


In [ ]:
teardown_opik()
print('chapter 04 done →', trace_info.get('jsonl_path'))

## 다음 챕터 (Ch 5 — Debate Pool)
Ch 4 의 답변은 "단일 모델 + 인용 강제". Ch 5에서는 4 provider 모두를 동시에 debate 라운드에 참여시키고 self-reflect + debate 하이브리드 전략의 trade-off를 측정한다.